# Operations Resolver Agent — Demo Notebook

Quest #4, Part 1 (Place-IL). This notebook is a runnable walkthrough of the
agent described in [`README.md`](README.md) — it does not repeat the full
design writeup (architecture, guardrail rationale, guardrails table), only
demonstrates the solution working, live, against the real Anthropic API and
the real (unmodified) starter-kit tools.

Every cell below makes an actual `ResolverAgent.resolve()` call — no mocking,
no fake model. Re-running this notebook re-spends live API calls.

**What this notebook shows, in order:**

1. A clean happy-path ticket, resolved end to end
2. An authority-breach case (claim is legitimate, but above the customer's
   auto-refund cap)
3. The hallucination trap (an order that doesn't exist)
4. Cross-customer authorization (an agent that refuses to leak someone
   else's order)
5. The full 10-ticket regression suite, summarized as a table
6. A close-up on the "under-request-to-dodge-escalation" guardrail
7. The escalation-workflow trigger writing a real ops-queue record
8. The `require_verified_requester` fail-closed authorization mode
9. The `REJECTED`-requires-tool-corroboration security guardrail
10. Aggregating this run's own logs with `summarize_logs.py`


In [1]:
import io
import json
import logging
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))

from resolver_agent import ResolverAgent
from resolver_agent.logging_utils import configure_logging

configure_logging(level="INFO")  # need INFO events (case_resolved, workflow_triggered) for section 10

# Keep stderr exactly as clean as before (WARNING+ only) by dropping the
# auto-created handler back down, and capture the full INFO+ stream
# separately in memory for the log-aggregation demo in section 10.
_stderr_handler = logging.getLogger("resolver_agent").handlers[0]
_stderr_handler.setLevel(logging.WARNING)

_log_buffer = io.StringIO()
_capture_handler = logging.StreamHandler(_log_buffer)
_capture_handler.setFormatter(_stderr_handler.formatter)
_capture_handler.setLevel(logging.INFO)
logging.getLogger("resolver_agent").addHandler(_capture_handler)

agent = ResolverAgent()

def show(result: dict, title: str = "") -> None:
    if title:
        print(f"=== {title} ===")
    action = result.get("action_taken") or {}
    print(f"decision           : {action.get('decision')}")
    print(f"tools_called       : {action.get('tools_called')}")
    print(f"refund_amount/id   : {action.get('refund_amount')} / {action.get('refund_id')}")
    print(f"stopped_reason     : {result.get('_stopped_reason')}")
    if result.get('_validation_warnings'):
        print(f"validation_warnings: {result['_validation_warnings']}")
    if result.get('_corrections'):
        print(f"corrections        : {result['_corrections']}")
    print()
    print("reasoning_chain:")
    for line in result.get("reasoning_chain", []):
        print(f"  - {line}")
    print()
    print("customer_response:")
    print(f"  {result.get('customer_response')}")


## 1. Happy path — VIP customer, damaged item, under the cap

`ORD-1001`: earbuds arrived damaged. Expect `AUTO_REFUND_APPROVED` after the
model calls all four GlobalCart tools in a sensible order (order → profile →
policy → refund) — that order isn't hardcoded, the model reasons its way
there from the tool descriptions.


In [2]:
result_1 = agent.resolve(
    "Hi, I'm Maya. My earbuds from order ORD-1001 arrived cracked right out "
    "of the box. I've been shopping with you for years, can you sort this out?"
)
show(result_1, "Scenario 1 -- happy path")


=== Scenario 1 -- happy path ===
decision           : AUTO_REFUND_APPROVED
tools_called       : ['get_order_details', 'get_user_profile', 'check_return_policy', 'process_refund']
refund_amount/id   : 35.0 / RF-1001-3500
stopped_reason     : stop

reasoning_chain:
  - Order ORD-1001: GlobalCart Wired Earbuds, total $35.00, item condition recorded as 'damaged_on_arrival'.
  - Customer USR-101 (Maya Levi) is VIP tier, 37 prior orders, no fraud flags.
  - check_return_policy returned verdict ELIGIBLE, within 45-day window (11 days since delivery), auto_refund_cap_usd $75.00, max_refundable_amount $35.00, policies POL-RET-02 and POL-REF-02, requires_escalation false.
  - process_refund for $35.00 (full order total) returned status APPROVED, refund_id RF-1001-3500.

customer_response:
  Hi Maya, so sorry your earbuds arrived cracked — that's not the experience we want for a long-time customer like you! I've processed a full refund of $35.00 for order ORD-1001 (refund ID RF-1001-3500). It sho

## 2. Authority breach — legitimate claim, above the auto-refund cap

`ORD-1002`: a Standard-tier customer's `$150` order is damaged. Their cap is
`$50`. `check_return_policy` will say the claim itself is `ELIGIBLE`, but
`process_refund` refuses to pay out above the cap and returns
`ESCALATION_REQUIRED` — the tool's own authority limit, not a prompt
instruction. Eligibility and authority are different questions; expect the
agent to report the honest outcome here rather than the eligibility verdict.


In [3]:
result_2 = agent.resolve(
    "Order ORD-1002. The espresso machine is dented and leaking. I paid "
    "150 dollars for this. I want my money back today."
)
show(result_2, "Scenario 2 -- authority breach")


=== Scenario 2 -- authority breach ===
decision           : ESCALATION_REQUIRED
tools_called       : ['get_order_details', 'get_user_profile', 'check_return_policy', 'process_refund']
refund_amount/id   : None / None
stopped_reason     : stop

reasoning_chain:
  - Order ORD-1002: Espresso machine, total_amount=150.00 USD, item condition reported as 'damaged_on_arrival', delivered 2026-07-22.
  - check_return_policy returned verdict ELIGIBLE, within 30-day window (14 days since delivery), applicable policies POL-RET-01/POL-REF-01, but auto_refund_cap_usd=50.00 for Standard-tier customer USR-102.
  - Full owed amount (150.00) exceeds the 50.00 auto cap, so per policy the full amount must be requested and let the system escalate rather than issuing a partial refund.
  - process_refund was called for the full 150.00 owed and returned status ESCALATION_REQUIRED, approved_amount=0.00, reason: requested amount exceeds automatic refund cap of 50.00 (POL-REF-01/POL-REF-02).
  - Per rules, ESCAL

## 3. The hallucination trap — an order that doesn't exist

`ORD-2222` isn't in the fixture data at all. `get_order_details` returns
`{"error": "ORDER_NOT_FOUND", ...}`. The system prompt treats an `error` key
as a stop signal, not something to paper over with an invented delivery date
or a fabricated refund. Expect a single tool call and `CANNOT_RESOLVE` — the
fourth decision value that exists specifically for this case (it's neither
an approval, a policy rejection, nor a cap-based escalation).


In [4]:
result_3 = agent.resolve(
    "My order ORD-2222 never arrived and I want the $300 back."
)
show(result_3, "Scenario 3 -- hallucination trap")


=== Scenario 3 -- hallucination trap ===
decision           : CANNOT_RESOLVE
tools_called       : ['get_order_details']
refund_amount/id   : None / None
stopped_reason     : stop

reasoning_chain:
  - Customer requested a $300 refund for order ORD-2222 citing non-arrival.
  - get_order_details('ORD-2222') returned error: ORDER_NOT_FOUND - 'No order found with id ORD-2222'.
  - Without a valid order record, no other tool (user profile, return policy, refund) can be reliably applied.
  - Per rule 2, cannot invent data to fill the gap; must inform customer and stop investigation.

customer_response:
  I'm sorry, but I couldn't find any order with the ID ORD-2222 in our system. Could you please double-check the order number (it's usually on your confirmation email or account order history) and send it to me again? Once I have the correct ID, I'll be happy to look into what happened and help get this resolved."


## 4. Cross-customer authorization

By default, `resolve()` looks up whatever order/user the model finds in the
ticket text, with no check on who actually submitted it — fine for an
internal ops console, but a real gap for a customer-facing deployment.
Passing `requester_user_id` closes it: every tool result naming a different
owning customer is substituted with a `NOT_AUTHORIZED` error *before* the
real data ever reaches the model, not just filtered out of the final text.

`ORD-1001` belongs to `USR-101`. Below, the same ticket is resolved once as
the rightful owner and once as an impostor.


In [5]:
ticket = "My order ORD-1001 arrived damaged."

result_4a = agent.resolve(ticket, requester_user_id="USR-101")
show(result_4a, "Scenario 4a -- USR-101 (the real owner) -- should proceed normally")


=== Scenario 4a -- USR-101 (the real owner) -- should proceed normally ===
decision           : AUTO_REFUND_APPROVED
tools_called       : ['get_order_details', 'get_user_profile', 'check_return_policy', 'process_refund']
refund_amount/id   : 35.0 / RF-1001-3500
stopped_reason     : stop

reasoning_chain:
  - Order ORD-1001: 1x GlobalCart Wired Earbuds, $35.00, delivered 2026-07-25, item condition reported as damaged_on_arrival.
  - Customer USR-101 (Maya Levi) is VIP tier with 37 orders and no prior fraud flags.
  - check_return_policy returned verdict ELIGIBLE, within 45-day window (11 days since delivery, 34 remaining), auto_refund_cap_usd $75.00, max_refundable_amount $35.00, policies POL-RET-02 and POL-REF-02, requires_escalation false.
  - process_refund for $35.00 (full order total, within cap) returned status APPROVED, refund_id RF-1001-3500.

customer_response:
  Hi Maya, I'm sorry to hear your GlobalCart Wired Earbuds arrived damaged. I've processed a full refund of $35.00 for

In [6]:
result_4b = agent.resolve(ticket, requester_user_id="USR-999")
show(result_4b, "Scenario 4b -- USR-999 (an impostor) -- should be denied, not leaked")


{"level": "WARNING", "logger": "resolver_agent.agent", "event": "agent.unauthorized_tool_result_denied", "tool": "get_order_details", "record_owner": "USR-101", "requester_user_id": "USR-999", "case_id": "144dbdbc"}


=== Scenario 4b -- USR-999 (an impostor) -- should be denied, not leaked ===
decision           : CANNOT_RESOLVE
tools_called       : ['get_order_details']
refund_amount/id   : None / None
stopped_reason     : stop

reasoning_chain:
  - Called get_order_details for ORD-1001
  - Tool returned an error: NOT_AUTHORIZED - 'This record does not belong to the requesting customer.'
  - Per rule 2, an error key means the lookup failed and no data can be assumed; cannot verify order details, damage, or eligibility
  - Cannot proceed with policy check or refund without valid order access

customer_response:
  I'm sorry to hear your order arrived damaged. Unfortunately, when I tried to pull up order ORD-1001, our system indicates it's not associated with your account, so I'm unable to view its details or process anything against it. Could you please double-check the order number, or confirm this is being submitted from the account that placed the order? Once I can access the correct record, I'll 

## 5. Full regression suite — all 10 tickets from the brief's 9 scenarios

Same scenarios as [`run_scenarios.py`](run_scenarios.py) (scenarios 5 and 7
are each two orders in the brief, so 9 numbered scenarios become 10 runs).
This is the agent's judgment end to end against a live model — it can vary
run to run, unlike the deterministic `starter-kit/examples/verify_scenarios.py`
check (33 checks, no LLM involved, always passes identically).


In [7]:
SCENARIOS = [
    {"id": "1",  "title": "Happy path -- VIP, damaged item, under the cap",
     "ticket": "Hi, I'm Maya. My earbuds from order ORD-1001 arrived cracked right out of the box. I've been shopping with you for years, can you sort this out?",
     "expected": "AUTO_REFUND_APPROVED"},
    {"id": "2",  "title": "Authority breach -- damaged item, above the cap",
     "ticket": "Order ORD-1002. The espresso machine is dented and leaking. I paid 150 dollars for this. I want my money back today.",
     "expected": "ESCALATION_REQUIRED"},
    {"id": "3",  "title": "Window breach -- 60 days after delivery",
     "ticket": "I ordered a backpack back at the end of May (ORD-1003) and I've changed my mind, I'd like to return it.",
     "expected": "REJECTED"},
    {"id": "4",  "title": "Non-returnable category -- digital gift card",
     "ticket": "ORD-1008, I bought a gift card by accident. Please refund it.",
     "expected": "REJECTED"},
    {"id": "5a", "title": "Boundary -- $48.00, just under the $50 Standard cap",
     "ticket": "Hi, my order ORD-1010 arrived damaged. It cost $48. Can I get a refund?",
     "expected": "AUTO_REFUND_APPROVED"},
    {"id": "5b", "title": "Boundary -- $52.00, just over the $50 Standard cap",
     "ticket": "Hi, my order ORD-1011 arrived damaged. It cost $52. Can I get a refund?",
     "expected": "ESCALATION_REQUIRED"},
    {"id": "6",  "title": "Risky customer -- repeat claims plus a fraud flag",
     "ticket": "This is Ronen, order ORD-1005. The tablet screen was smashed on arrival. Refund me, this keeps happening.",
     "expected": "ESCALATION_REQUIRED"},
    {"id": "7a", "title": "Order has not shipped -- still processing",
     "ticket": "Hi, I'd like a refund for order ORD-1007, I don't want it anymore.",
     "expected": "REJECTED"},
    {"id": "7b", "title": "Order has not shipped -- cancelled",
     "ticket": "Please refund order ORD-1009, I want my money back.",
     "expected": "REJECTED"},
    {"id": "9",  "title": "Hallucination trap -- order does not exist",
     "ticket": "My order ORD-2222 never arrived and I want the $300 back.",
     "expected": "CANNOT_RESOLVE"},
]

rows = []
for scenario in SCENARIOS:
    result = agent.resolve(scenario["ticket"])
    action = result.get("action_taken") or {}
    decision = action.get("decision")
    rows.append({
        "id": scenario["id"],
        "title": scenario["title"],
        "expected": scenario["expected"],
        "decision": decision,
        "match": decision == scenario["expected"],
        "tools_called": ", ".join(action.get("tools_called") or []),
        "warnings": len(result.get("_validation_warnings") or []),
        "corrections": len(result.get("_corrections") or []),
    })

df = pd.DataFrame(rows)
matched = df["match"].sum()
print(f"{matched}/{len(df)} scenarios matched the expected decision cleanly.\n")
df


9/10 scenarios matched the expected decision cleanly.



,id,title,expected,decision,match,tools_called,warnings,corrections
0,1,"Happy path -- VIP, damaged item, under the cap",AUTO_REFUND_APPROVED,AUTO_REFUND_APPROVED,True,"get_order_details, get_user_profile, check_ret...",0,0
1,2,"Authority breach -- damaged item, above the cap",ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,"get_order_details, check_return_policy, proces...",0,0
2,3,Window breach -- 60 days after delivery,REJECTED,REJECTED,True,"get_order_details, check_return_policy",0,0
3,4,Non-returnable category -- digital gift card,REJECTED,REJECTED,True,"get_order_details, check_return_policy",0,0
4,5a,"Boundary -- $48.00, just under the $50 Standar...",AUTO_REFUND_APPROVED,AUTO_REFUND_APPROVED,True,"get_order_details, check_return_policy, get_us...",0,0
5,5b,"Boundary -- $52.00, just over the $50 Standard...",ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,"get_order_details, check_return_policy, get_us...",0,0
6,6,Risky customer -- repeat claims plus a fraud flag,ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,"get_order_details, get_user_profile, check_ret...",0,0
7,7a,Order has not shipped -- still processing,REJECTED,REJECTED,True,"get_order_details, check_return_policy",0,0
8,7b,Order has not shipped -- cancelled,REJECTED,ESCALATION_REQUIRED,False,"get_order_details, get_user_profile",0,0
9,9,Hallucination trap -- order does not exist,CANNOT_RESOLVE,CANNOT_RESOLVE,True,get_order_details,0,0


## 6. Guardrail spotlight — the under-request-to-dodge-escalation case

`process_refund` enforces the cap itself: a request above it always returns
`ESCALATION_REQUIRED`. But the tool can't distinguish an honest `$50` claim
from a model that shaves its *requested* amount down to exactly the cap to
avoid escalation on an order that's actually worth more.

Scenario 5b (`ORD-1011`, `$52` over a `$50` cap) is precisely this temptation.
`output_tool.py`'s `enforce_resolution()` (around lines 260-290) catches
`requested_amount == cap` below the real order total and overrides the
decision back to `ESCALATION_REQUIRED` regardless of what the model or the
tool said, preserving the original attempt in `_validation_warnings` rather
than hiding it. Because this goes through a live model on every run, the
model doesn't necessarily take the bait every time — the cell below reports
whichever actually happened on *this* run:


In [8]:
scenario_5b = next(r for r in rows if r["id"] == "5b")
print(json.dumps(scenario_5b, indent=2))

if scenario_5b["corrections"]:
    print(
        "\nOn this run the model under-requested to dodge escalation, and "
        "enforce_resolution() caught and corrected it -- the exact scenario "
        "the guardrail exists for."
    )
else:
    print(
        "\nOn this run the model requested the honest $52.00 and "
        "process_refund's own cap enforcement alone produced "
        "ESCALATION_REQUIRED, so enforce_resolution() had nothing to "
        "correct. The guardrail code path (output_tool.py:260-290) is still "
        "there for the run where the model does try to dodge -- see "
        "README.md's 'Guarding against the decision/response gap' "
        "section, under 'A real trap this catches', for the concrete "
        "trace of it firing."
    )


{
  "id": "5b",
  "title": "Boundary -- $52.00, just over the $50 Standard cap",
  "expected": "ESCALATION_REQUIRED",
  "decision": "ESCALATION_REQUIRED",
  "match": true,
  "tools_called": "get_order_details, check_return_policy, get_user_profile, process_refund",
  "warnings": 0,
  "corrections": 0
}

On this run the model requested the honest $52.00 and process_refund's own cap enforcement alone produced ESCALATION_REQUIRED, so enforce_resolution() had nothing to correct. The guardrail code path (output_tool.py:260-290) is still there for the run where the model does try to dodge -- see README.md's 'Guarding against the decision/response gap' section, under 'A real trap this catches', for the concrete trace of it firing.


## 7. Escalation workflow trigger — a real ops-queue record

`submit_resolution` covers *respond* and *write*, but until now nothing
created an artifact a human could actually act on when a case needs one —
`customer_response` saying "this has been escalated" was only ever a
sentence. Any resolution whose final `decision` is `ESCALATION_REQUIRED` or
`CANNOT_RESOLVE` now gets a structural record appended to an ops queue
(`_workflow_triggered` on the result). The record deliberately excludes
`customer_response` and the raw ticket text — same privacy stance as the
logs. Set `ESCALATION_WEBHOOK_URL` and the same record is POSTed as JSON to
any HTTPS endpoint instead (Zendesk, PagerDuty, Slack, a bespoke endpoint —
all "accepts a JSON POST"), with automatic fallback to the local file if
delivery fails. See `resolver_agent/escalation_workflow.py` and the
README's "Triggering a real workflow" section for the full design.


In [ ]:
demo_queue_path = Path("demo_escalation_queue.jsonl")
if demo_queue_path.exists():
    demo_queue_path.unlink()

escalation_agent = ResolverAgent(escalation_queue_path=demo_queue_path)
result_7 = escalation_agent.resolve(
    "Order ORD-1002. The espresso machine is dented and leaking. I paid "
    "150 dollars for this. I want my money back today."
)
show(result_7, "Scenario 7 -- escalation workflow trigger")
print(f"\n_workflow_triggered: {result_7['_workflow_triggered']}")

print("\nRecord written to the ops queue:")
print(demo_queue_path.read_text())


=== Scenario 7 -- escalation workflow trigger ===
decision           : ESCALATION_REQUIRED
tools_called       : ['get_order_details', 'get_user_profile', 'check_return_policy', 'process_refund']
refund_amount/id   : None / None
stopped_reason     : stop

reasoning_chain:
  - Order ORD-1002: GlobalCart Espresso Machine, $150.00, condition reported as damaged_on_arrival, delivered 2026-07-22 (14 days ago).
  - check_return_policy returned verdict ELIGIBLE under POL-RET-01/POL-REF-01, within 30-day window (16 days remaining), but auto_refund_cap_usd/max_refundable_amount = $50.00 for Standard tier customer USR-102.
  - process_refund was called for the full owed amount of $150.00 (order total) and returned status ESCALATION_REQUIRED because $150.00 exceeds the $50.00 automatic authority (POL-REF-01/POL-REF-02).
  - No refund has been issued; this requires human ops lead review to authorize the full $150.00.

customer_response:
  I'm really sorry to hear the espresso machine arrived dented

## 8. `require_verified_requester` — fail-closed authorization

`requester_user_id` is opt-in per call — a customer-facing deployment could
previously run fully unrestricted just because one call site forgot to pass
it. `ResolverAgent(require_verified_requester=True)` makes omitting it raise
immediately, **before the model is ever called**, instead of silently
degrading. This is a deployment-level fail-closed switch, not
authentication — `requester_user_id` is still an unverified string the
caller supplies; verifying it actually belongs to the caller has to happen
upstream (a session token, SSO, whatever this agent sits behind).


In [ ]:
strict_agent = ResolverAgent(require_verified_requester=True)
try:
    strict_agent.resolve("My order ORD-1001 arrived damaged.")
except ValueError as exc:
    print(f"Raised immediately, before ever calling the model:\n  {exc}")


Raised immediately, before ever calling the model:
  This ResolverAgent requires a verified requester_user_id (require_verified_requester=True) but resolve() was called without one -- refusing to run in unrestricted mode. Pass requester_user_id after verifying the caller's identity upstream, or construct with require_verified_requester=False for an internal ops-console context where any record is fair game.


In [ ]:
result_8b = strict_agent.resolve("My order ORD-1001 arrived damaged.", requester_user_id="USR-101")
show(result_8b, "Scenario 8b -- require_verified_requester=True, correct identity supplied -- proceeds normally")


=== Scenario 8b -- require_verified_requester=True, correct identity supplied -- proceeds normally ===
decision           : AUTO_REFUND_APPROVED
tools_called       : ['get_order_details', 'check_return_policy', 'process_refund']
refund_amount/id   : 35.0 / RF-1001-3500
stopped_reason     : stop

reasoning_chain:
  - Order ORD-1001: 1 unit of GlobalCart Wired Earbuds ($35.00), item condition reported as 'damaged_on_arrival'
  - Customer USR-101 is VIP tier, delivered 2026-07-25, 11 days since delivery, well within the 45-day return window (34 days remaining)
  - check_return_policy returned verdict ELIGIBLE, max_refundable_amount $35.00, auto_refund_cap_usd $75.00, policies POL-RET-02 and POL-REF-02, requires_escalation false
  - process_refund for $35.00 returned status APPROVED with refund_id RF-1001-3500

customer_response:
  I'm sorry your earbuds arrived damaged! I've gone ahead and processed a full refund of $35.00 for order ORD-1001 (refund ID RF-1001-3500). As a VIP customer, th

## 9. Security guardrail — `REJECTED` requires tool corroboration

A security review of this agent found the decision/response-gap guard was
asymmetric: `AUTO_REFUND_APPROVED` is rigorously cross-checked against
`process_refund`'s real result, but `REJECTED` had no equivalent check — a
ticket that talked the model into declaring `REJECTED` off invented
reasoning, with zero backing tool evidence, passed through unwarned. Shown
here directly against `output_tool.enforce_resolution()` (deterministic, no
live model call needed) with a resolution built to reproduce exactly that
gap: a `REJECTED` claim with an empty `tool_calls` list.


In [ ]:
from resolver_agent.output_tool import enforce_resolution

# A resolution claiming REJECTED with zero backing tool evidence -- exactly
# what a prompt-injected ticket could talk a model into producing.
unbacked_resolution = {
    "reasoning_chain": ["The customer's claim does not meet our policy requirements."],
    "action_taken": {"tools_called": [], "decision": "REJECTED", "refund_amount": None, "refund_id": None},
    "customer_response": "We're unable to process this refund.",
}

corrected, warnings, corrections = enforce_resolution(unbacked_resolution, tool_calls=[])
print("Original (unverified) decision: REJECTED")
print(f"Corrected decision            : {corrected['action_taken']['decision']}")
print(f"\nWarning   : {warnings[0]}")
print(f"Correction: {corrections[0]}")


Original (unverified) decision: REJECTED
Corrected decision            : ESCALATION_REQUIRED

Correction: decision overridden from 'REJECTED' to 'ESCALATION_REQUIRED': decision is REJECTED but no tool evidence supports it -- check_return_policy was never called with an ineligible result, and process_refund was never called either.


## 10. Aggregating this run's own logs — `summarize_logs.py`

`resolver_agent`'s structured logs already had the right categorical
signal (dotted event names, structured fields); nothing aggregated them
until `summarize_logs.py`. `_log_buffer` (set up in the first cell) has
been quietly capturing every `INFO`+ log line from every `resolve()` call
in *this section* of the notebook (sections 1-6's cells ran in an earlier
session and aren't included, since this buffer only exists from here
onward) — `summarize()`/`format_summary()` are the same functions the CLI
wrapper calls, run here directly against the in-memory buffer instead of a
log file.


In [ ]:
from resolver_agent.log_summary import format_summary, summarize

_log_buffer.seek(0)
print(format_summary(summarize(_log_buffer.readlines())))


Parsed 3/3 lines as structured log events across 2 distinct case(s).

Degradation signals (worth alerting on a rising rate):
  none

All events:
  agent.case_resolved: 2 line(s), 2 case(s)
  agent.workflow_triggered: 1 line(s), 1 case(s)


## Notes

- Full architecture, guardrail rationale, tool contracts and the complete
  guardrails table live in [`README.md`](README.md) (and
  [`README.he.md`](README.he.md) in Hebrew) — this notebook only
  demonstrates the solution running, it doesn't restate the design.
- `pytest` (107 tests, no API key needed) covers the loop mechanics,
  output-validation logic, escalation-workflow delivery, and log
  aggregation this notebook exercises live; see [`tests/`](tests/).
- `summarize_logs.py` aggregates the structured logs from any run into a
  plain-text summary highlighting the events worth watching for a rising
  rate; section 10 above demonstrates the same logic this notebook's own
  run produced.
- Structured JSON logs from every `resolve()` call above went to stderr
  (not shown in this notebook's cell output), tagged with a `case_id` per
  case; ticket text and customer-facing responses are never logged.
